In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from pymongo import MongoClient
import yaml
from bson import json_util
from pydantic import BaseModel
from typing import Dict


In [2]:
class BackUpConfig(BaseModel):
    db: str
    url: str
    backup_folder: str
    collections: Dict[str, str]
    year: str


def load_config(path: str) -> BackUpConfig:
    with open(path, "r") as f:
        raw = yaml.safe_load(f)

    return BackUpConfig(
            db=raw["mongo"]["db"],
            url=raw["mongo"]["url"],
            backup_folder=raw["mongo"]["backup_folder"],
            collection=raw["mongo"]["collection"],
            year=raw["season"]["year"]

        )

events_config = load_config("../config/config.yaml")


In [3]:

backup_root = Path(events_config.backup_folder)
backup_root.mkdir(parents=True, exist_ok=True)

client = MongoClient(events_config.url)
db = client[events_config.db]

season = events_config.year
collections = {
    "schedule": events_config.collections["collection_schedule"],
    "raw_events": events_config.collections["collection_raw_events"],
}

for name, coll_name in collections.items():
    coll = db[coll_name]
    docs = list(coll.find({'season': season}))

    if not docs:
        print(f"No records found for season {season} in {name}")
        continue

    out_json_name = f"{season}_{name}.json"
    out_zip = backup_root / f"{season}_{name}.zip"

    json_text = json_util.dumps(docs, indent=2)

    with ZipFile(out_zip, "w", compression=ZIP_DEFLATED) as zf:
        zf.writestr(out_json_name, json_text.encode("utf-8"))

    print(f"Season: {season} -> Saved {len(docs)} documents for {name} and zipped to {out_zip}")

Season: 2025-2026 -> Saved 279 documents for schedule and zipped to mongo_backup/2025-2026_schedule.zip
Season: 2025-2026 -> Saved 436885 documents for raw_events and zipped to mongo_backup/2025-2026_raw_events.zip


### Index Preparation

In [1]:
# from pymongo import MongoClient

# client = MongoClient("mongodb://localhost:27017/")
# db = client["WhoScored"]

# collections = [
#     "available_teams",
#     "game_schedule",
#     "game_raw_events",
#     "game_processed_events",
#     "game_team_stats",
#     "game_player_stats",
#     "game_shot_sequences",
#     "game_pass_sequences",
# ]

# for collection_name in collections:
#     collection = db[collection_name]
#     print(f"\n{collection_name}")

#     before = list(collection.list_indexes())
#     custom_indexes = [idx["name"] for idx in before if idx["name"] != "_id_"]

#     if not custom_indexes:
#         print("  no custom indexes to delete")
#         continue

#     for index_name in custom_indexes:
#         collection.drop_index(index_name)
#         print(f"  deleted: {index_name}")

# client.close()

In [2]:
# from pymongo import MongoClient, ASCENDING

# client = MongoClient("mongodb://localhost:27017/")
# db = client["WhoScored"]

# indexes = {
#     "available_teams": [
#         ([("ws_team_id", ASCENDING)], {"unique": True, "name": "uniq_ws_team_id"}),
#         ([("ws_team_name", ASCENDING)], {"name": "idx_ws_team_name"}),
#         ([("fbref_team_name", ASCENDING)], {"name": "idx_fbref_team_name"}),
#     ],

#     "game_schedule": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"unique": True, "name": "uniq_season_game"}),
#         ([("season", ASCENDING), ("game_status", ASCENDING)], {"name": "idx_season_status"}),
#         ([("season", ASCENDING), ("game_date", ASCENDING)], {"name": "idx_season_game_date"}),
#         ([("season", ASCENDING), ("home_team_id", ASCENDING)], {"name": "idx_season_home_team"}),
#         ([("season", ASCENDING), ("away_team_id", ASCENDING)], {"name": "idx_season_away_team"}),
#     ],

#     "game_raw_events": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         ([("season", ASCENDING), ("game_id", ASCENDING), ("event_idx", ASCENDING)], {"unique": True, "name": "uniq_season_game_event"}),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#         (
#             [("season", ASCENDING), ("player_id", ASCENDING)],
#             {"name": "idx_season_player", "partialFilterExpression": {"player_id": {"$type": "number"}}},
#         ),
#     ],

#     "game_processed_events": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         ([("season", ASCENDING), ("game_id", ASCENDING), ("event_idx", ASCENDING)], {"unique": True, "name": "uniq_season_game_event"}),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#         (
#             [("season", ASCENDING), ("player_id", ASCENDING)],
#             {"name": "idx_season_player", "partialFilterExpression": {"player_id": {"$type": "number"}}},
#         ),
#         ([("season", ASCENDING), ("type", ASCENDING)], {"name": "idx_season_type"}),
#         ([("season", ASCENDING), ("qualifier_ids", ASCENDING)], {"name": "idx_season_qualifier_ids"}),
#     ],

#     "game_team_stats": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         ([("season", ASCENDING), ("game_id", ASCENDING), ("team_id", ASCENDING)], {"unique": True, "name": "uniq_season_game_team"}),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#         ([("season", ASCENDING), ("opponent_team_id", ASCENDING)], {"name": "idx_season_opponent"}),
#     ],

#     "game_player_stats": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         (
#             [("season", ASCENDING), ("game_id", ASCENDING), ("player_id", ASCENDING)],
#             {
#                 "unique": True,
#                 "name": "uniq_season_game_player",
#                 "partialFilterExpression": {"player_id": {"$type": "number"}},
#             },
#         ),
#         (
#             [("season", ASCENDING), ("player_id", ASCENDING)],
#             {"name": "idx_season_player", "partialFilterExpression": {"player_id": {"$type": "number"}}},
#         ),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#     ],

#     "game_shot_sequences": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         ([("season", ASCENDING), ("game_id", ASCENDING), ("sequence_key", ASCENDING), ("sequence_event", ASCENDING)], {"unique": True, "name": "uniq_shot_sequence_event"}),
#         ([("season", ASCENDING), ("sequence_key", ASCENDING)], {"name": "idx_season_sequence_key"}),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#         (
#             [("season", ASCENDING), ("player_id", ASCENDING)],
#             {"name": "idx_season_player", "partialFilterExpression": {"player_id": {"$type": "number"}}},
#         ),
#     ],

#     "game_pass_sequences": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {"name": "idx_season_game"}),
#         ([("season", ASCENDING), ("game_id", ASCENDING), ("sequence_key", ASCENDING), ("sequence_event", ASCENDING)], {"unique": True, "name": "uniq_pass_sequence_event"}),
#         ([("season", ASCENDING), ("sequence_key", ASCENDING)], {"name": "idx_season_sequence_key"}),
#         ([("season", ASCENDING), ("sequence_team_id", ASCENDING)], {"name": "idx_season_sequence_team"}),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {"name": "idx_season_team"}),
#         (
#             [("season", ASCENDING), ("player_id", ASCENDING)],
#             {"name": "idx_season_player", "partialFilterExpression": {"player_id": {"$type": "number"}}},
#         ),
#     ],
# }

# for collection_name, specs in indexes.items():
#     collection = db[collection_name]
#     print(f"\n{collection_name}")

#     for keys, options in specs:
#         try:
#             index_name = collection.create_index(keys, **options)
#             print(f"  OK: {index_name}")
#         except Exception as exc:
#             print(f"  FAILED: {options.get('name')} -> {exc}")

# client.close()